# Digital Projects: Data Cleaning & Quality Assessment

**Project:** Digital Analytics & Insights — Programme Monitoring  
**Stage:** Data preparation and quality assessment  
**Tools:** Python, pandas

## Project context

This notebook prepares a simulated digital-project dataset for programme monitoring and subsequent analysis in PostgreSQL and Power BI.

The workflow covers:
- Initial inspection and profiling
- Duplicate identification
- Standardization of categorical values
- Date and progress validation
- Review of missing values and financial exceptions
- Export of a clean dataset with data-quality flags retained

> **Data note:** This is an independent portfolio exercise using simulated data. It is not an official UNDP dataset or assignment.

## Before running

Place the source CSV at `data/raw/digital_projects.csv` in the repository, and save this notebook in the `notebooks/` folder. The cleaned output will be written to `data/processed/digital_projects_clean.csv`.

Required package: `pandas`.


## 1. Import libraries and load the dataset

In [ ]:
from pathlib import Path

import pandas as pd

# Repository-relative paths: no personal computer paths or usernames.
# Support execution from either the repository root or notebooks/.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (candidate for candidate in CANDIDATE_ROOTS
     if (candidate / "data" / "raw" / "digital_projects.csv").exists()),
    Path.cwd().parent
)
INPUT_FILE = PROJECT_ROOT / "data" / "raw" / "digital_projects.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "digital_projects_clean.csv"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Source dataset not found: {INPUT_FILE}\n"
        "Place digital_projects.csv in the data/raw/ folder."
    )

df = pd.read_csv(INPUT_FILE)

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


## 2. Initial data inspection

In [ ]:
# Review column names, data types, and non-null counts.
df.info()


In [ ]:
# Inspect the dataset dimensions and column names.
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
# Review missing values by column.
missing_values = (
    df.isna()
      .sum()
      .rename("missing_count")
      .to_frame()
)
missing_values["missing_pct"] = (
    missing_values["missing_count"] / len(df) * 100
).round(2)

missing_values.sort_values("missing_count", ascending=False)


In [ ]:
# Check full-row duplicates and duplicate project identifiers separately.
print("Exact duplicate rows:", df.duplicated().sum())

if "Project_ID" in df.columns:
    print("Duplicate Project_ID values:", df["Project_ID"].duplicated().sum())


## 3. Identify and handle duplicate project records

The project identifier is expected to uniquely identify a project. Review duplicate IDs before removing records.

The following cell displays all rows associated with duplicate IDs. The cleaning step retains the first occurrence of each `Project_ID`; this is appropriate only after confirming that the retained record is the intended record for this exercise.


In [ ]:
duplicate_projects = df[
    df["Project_ID"].duplicated(keep=False)
].sort_values("Project_ID")

duplicate_projects


In [ ]:
# Remove duplicate project identifiers, retaining the first occurrence.
rows_before = len(df)
df = df.drop_duplicates(subset="Project_ID", keep="first").copy()
rows_removed = rows_before - len(df)

print(f"Rows before: {rows_before}")
print(f"Duplicate project records removed: {rows_removed}")
print(f"Rows after: {len(df)}")
print("Project_ID values are unique:", df["Project_ID"].is_unique)


## 4. Standardize categorical fields

In [ ]:
# Remove leading/trailing whitespace from text columns.
text_columns = df.select_dtypes(include="object").columns
for column in text_columns:
    df[column] = df[column].str.strip()

# Standardize country names and known formatting variants.
df["Country"] = df["Country"].str.title()

# Standardize status labels without merging genuinely different statuses.
df["Status"] = df["Status"].replace({
    "On-track": "On Track"
})

# Standardize the known digital-pillar capitalization variant.
df["Digital_Pillar"] = df["Digital_Pillar"].replace({
    "Digital skills": "Digital Skills"
})


In [ ]:
# Inspect the resulting categories to confirm standardization.
for column in ["Country", "Region", "Initiative", "Digital_Pillar", "Status"]:
    print(f"\n{column}:")
    print(sorted(df[column].dropna().unique().tolist()))


## 5. Validate dates and flag date logic issues

In [ ]:
# Convert date fields to datetime.
# Invalid date strings become NaT and can be investigated separately.
df["Start_Date"] = pd.to_datetime(df["Start_Date"], errors="coerce")
df["End_Date"] = pd.to_datetime(df["End_Date"], errors="coerce")

print(df[["Start_Date", "End_Date"]].dtypes)


In [ ]:
# Flag records where the end date precedes the start date.
df["Date_Logic_Issue"] = (
    df["Start_Date"].notna()
    & df["End_Date"].notna()
    & (df["End_Date"] < df["Start_Date"])
)

date_exceptions = df.loc[
    df["Date_Logic_Issue"],
    ["Project_ID", "Country", "Start_Date", "End_Date", "Status"]
]

print("Projects with end dates before start dates:", len(date_exceptions))
date_exceptions


## 6. Review missing values and key numeric fields

In [ ]:
# Inspect records with missing beneficiary-related values.
beneficiary_missing = df[
    df["Beneficiaries"].isna()
    | df["Female_Beneficiaries_Pct"].isna()
]

beneficiary_missing[
    ["Project_ID", "Country", "Beneficiaries", "Female_Beneficiaries_Pct"]
]


Missing beneficiary values are retained as missing rather than replaced with zero. A missing value does not mean that no beneficiaries were reached.

The following summaries help identify unusual values. They are diagnostic checks; they do not automatically change the source values.


In [ ]:
numeric_columns = [
    "Planned_Progress_Pct",
    "Actual_Progress_Pct",
    "Budget_USD",
    "Spent_USD",
    "Beneficiaries",
    "Female_Beneficiaries_Pct",
    "Youth_Beneficiaries_Pct",
]

df[numeric_columns].describe().round(2)


## 7. Investigate progress and financial exceptions

In [ ]:
# Review projects with actual progress above 100%.
progress_over_100 = df.loc[
    df["Actual_Progress_Pct"] > 100,
    [
        "Project_ID",
        "Country",
        "Status",
        "Planned_Progress_Pct",
        "Actual_Progress_Pct",
    ],
]

progress_over_100


An actual progress value above 100% is flagged for investigation rather than automatically capped. The original value is retained so that the exception remains visible to downstream analysts.


In [ ]:
# Calculate budget utilization for exploratory review.
# This is a derived diagnostic metric, not a source column.
df["Budget_Utilization_Pct"] = (
    df["Spent_USD"] / df["Budget_USD"].where(df["Budget_USD"] != 0)
) * 100

completed_projects_utilization = (
    df.loc[
        df["Status"].eq("Completed"),
        ["Project_ID", "Budget_USD", "Spent_USD", "Budget_Utilization_Pct"],
    ]
    .sort_values("Budget_Utilization_Pct")
)

completed_projects_utilization.head(10)


In [ ]:
# Identify projects where expenditure exceeds the approved budget.
over_budget_projects = df.loc[
    df["Spent_USD"] > df["Budget_USD"],
    ["Project_ID", "Country", "Budget_USD", "Spent_USD"],
].copy()

over_budget_projects["Overspend_USD"] = (
    over_budget_projects["Spent_USD"] - over_budget_projects["Budget_USD"]
)

over_budget_projects


Overspending is treated as a financial exception for review. It is not silently corrected, because expenditure above budget may require operational context and management follow-up.


## 8. Create data-quality flags

In [ ]:
# Retain transparent flags for downstream filtering in SQL and Power BI.
df["Progress_Logic_Issue"] = (
    df["Actual_Progress_Pct"].notna()
    & (
        (df["Actual_Progress_Pct"] < 0)
        | (df["Actual_Progress_Pct"] > 100)
    )
)

# The temporary utilization metric was used for review only.
# It will be recalculated in SQL rather than stored in the cleaned CSV.
df = df.drop(columns=["Budget_Utilization_Pct"])

print("Date logic exceptions:", int(df["Date_Logic_Issue"].sum()))
print("Progress logic exceptions:", int(df["Progress_Logic_Issue"].sum()))


In [ ]:
# Review all records with at least one data-quality flag.
quality_exceptions = df.loc[
    df["Date_Logic_Issue"] | df["Progress_Logic_Issue"],
    [
        "Project_ID",
        "Country",
        "Status",
        "Start_Date",
        "End_Date",
        "Planned_Progress_Pct",
        "Actual_Progress_Pct",
        "Date_Logic_Issue",
        "Progress_Logic_Issue",
    ],
]

quality_exceptions


## 9. Final validation and export

In [ ]:
# Final structural and quality checks.
print(f"Final dataset shape: {df.shape}")
print("Unique project IDs:", df["Project_ID"].is_unique)
print("Exact duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
display(df.isna().sum())

print("\nData-quality flag counts:")
print(df[["Date_Logic_Issue", "Progress_Logic_Issue"]].sum())


In [ ]:
# Export the cleaned dataset.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset exported to: {OUTPUT_FILE.relative_to(PROJECT_ROOT)}")
print(f"Exported rows: {len(df):,}")
print(f"Exported columns: {len(df.columns):,}")


## 10. Handoff to PostgreSQL

The cleaned CSV is intended for loading into PostgreSQL for the next stage of analysis.

Key preparation decisions:
- Duplicate project identifiers were reviewed and reduced to one record per project.
- Formatting inconsistencies in selected categorical fields were standardized.
- Missing beneficiary values were retained.
- Invalid date sequences and out-of-range actual progress values were flagged, not silently corrected.
- Budget utilization was used for exploratory checks but excluded from the exported dataset so it can be calculated consistently in SQL.

**Next stage:** Load `data/processed/digital_projects_clean.csv` into PostgreSQL and continue with portfolio, financial, implementation, country, digital-pillar, inclusion, and data-quality analysis.
